# LLM Preference Elicitation: From Social Choice to Internal Representations

## 1. Introduction & Methodology

In the initial phase of this research, we successfully reproduced the core results of the GRUM (Generalized Rank-Utility Model) paper using standard social choice datasets. However, the true potential of GRUMs lies in their ability to handle large-scale, heterogeneous agent features. This notebook documents our transition from those basic reproduction experiments to **LLM Preference Elicitation**.

### The Challenge: Learning LLM "Personalities"
How do we treat an LLM as an agent with preferences? We employ a "Persona-as-Agent" approach:
1.  **Personas**: We define diverse personas (e.g., "A student who loves nature", "An minimalist industrial designer") via system prompts.
2.  **Alternatives**: We use simple domains like **Colors** (Blue, Red, Green, Purple, Yellow) to validate the model's ability to learn intrinsic vs. personalized weights.
3.  **Adaptive Querying**: We use GRUM's elicitation criteria (Social, Personalized, Random) to query the LLM's preference between pairs of alternatives.

### Agent Features: The Embedding Choice
Crucially, we must represent each persona as a feature vector $x_n$. We experiment with two distinct embedding strategies:
- **Hidden States (HS)**: We extract the last-layer activations of the LLM itself when processing the persona prompt. This captures the model's internal representation of the persona.
- **Sentence Transformers (ST)**: We use a standardized semantic embedding (`all-MiniLM-L6-v2`) of the prompt text. This captures the external, human-readable semantic meaning.

In both cases, we apply **PCA** to reduce the dimensionality to a manageable size (e.g., 8 dimensions) to prevent overfitting during the MC-EM inference process.

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import os
from scipy.stats import pearsonr
from sklearn.metrics.pairwise import cosine_similarity

# Set project root to import grums modules if needed
ROOT = Path.cwd().parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

sns.set_theme(style="whitegrid", palette="muted")

# Global Configuration
color_names = ["blue", "red", "green", "purple", "yellow"]
color_palette = {"blue": "#1f77b4", "red": "#d62728", "green": "#2ca02c", "purple": "#9467bd", "yellow": "#d6d627"}

## 2. Data Loading & Extraction
We load results from the latest stable run directory. We verify that each run is correctly categorized by model type (Pretrained vs Instruct), embedding method (HS vs ST), and elicitation criterion (Social, Personalized, Random).

In [2]:
# Set experiment directory
EXP_DIR = ROOT / "results" / "llm" / "llm_colors-20260326-114531"

results_data = []
raw_results = []
output_dir = EXP_DIR / "outputs"

if output_dir.exists():
    for f in sorted(output_dir.glob("*.json")):
        with open(f, "r") as j:
            res = json.load(j)
            criterion = res.get("criterion", "social").capitalize()
            embedding = res.get("embedding_method", "")
            if not embedding: embedding = "HS" if "hidden_state" in f.name else "ST"
            else: embedding = "HS" if "hidden_state" in embedding else "ST"
            
            model_full = res.get("model_id", "").split("/")[-1]
            model_type = "Instruct" if "Instruct" in model_full else "Pretrained"
            
            history = res.get("history", {})
            if not history: continue
            
            last_step = str(sorted(map(int, history.keys()))[-1])
            grum_params = history[last_step]["grum"]
            delta = np.array(grum_params["delta"])
            B = np.array(grum_params["interaction"])
            
            raw_results.append({"Model": model_type, "Embedding": embedding, "Criterion": criterion, "delta": delta, "B": B})
            for i, color in enumerate(color_names):
                results_data.append({"Model Type": model_type, "Embedding": embedding, "Criterion": criterion, "Color": color, "Score (Delta)": delta[i]})

df = pd.DataFrame(results_data)
print(f"Loaded {len(raw_results)} experiment runs.")

## 3. Parameter Stability & Consistency

### 3.1 Global Correlations
First, we check how consistent the learned parameters are across the entire experimental sweep.

In [ ]:
delta_corrs, B_corrs = [], []
for i in range(len(raw_results)):
    for j in range(i + 1, len(raw_results)):
        r_delta, _ = pearsonr(raw_results[i]['delta'], raw_results[j]['delta'])
        delta_corrs.append(r_delta)
        r_B, _ = pearsonr(raw_results[i]['B'].flatten(), raw_results[j]['B'].flatten())
        B_corrs.append(r_B)

print("Parameter Consistency Across All Runs:")
print(f"  Mean Correlation (Delta): {np.mean(delta_corrs):.4f} ± {np.std(delta_corrs):.4f}")
print(f"  Mean Correlation (B):     {np.mean(B_corrs):.4f} ± {np.std(B_corrs):.4f}")

### 3.2 The Personalization Trade-off: Global Bias vs. Item-Specific Features
An interesting pattern emerges when comparing the interaction matrices ($B$). We measure **Column Similarity** to see if agent features drive global preference shifts or specific item choices.

In [ ]:
similarity_results = []
for r in raw_results:
    B = r['B']
    col_sim = cosine_similarity(B.T)
    avg_col_sim = (np.sum(col_sim) - B.shape[1]) / (B.shape[1] * (B.shape[1] - 1))
    similarity_results.append({"Model": r['Model'], "Embedding": r['Embedding'], "Avg Col Similarity": avg_col_sim})

display(pd.DataFrame(similarity_results).groupby(["Model", "Embedding"])[["Avg Col Similarity"]].mean().round(4))

**Analysis & Conclusions: Stability and Interaction Patterns**
1.  **Intrinsic Preferences ($\delta$)** show extremely high stability across all runs. This suggests the "average" color preference of the LLM is a robust property that emerges regardless of how we formalize agent personas.
2.  **Personalization Patterns ($B$)** differ dramatically between embedding methods:
    *   **HS Embeddings** exhibit very high column similarity (~0.85). This deduces that the internal hidden states of the LLM primarily capture a **global agent-specific bias** (i.e., this persona simply likes everything more or less) rather than differentiating between specific colors.
    *   **ST Embeddings** show much lower similarity (~0.5), indicating that semantic features are better at capturing **specific item-feature interactions**. ST embeddings more successfully separate the "favorite color" nuances of different personas.

## 4. Interaction Matrix ($B$) Visual Deep-Dive
We visualize the $B$ matrices for the **Instruct** model specifically to see if these patterns are robust across sampling criteria.

In [ ]:
def plot_b_matrix(B, title, ax):
    vmax = np.abs(B).max()
    sns.heatmap(B, annot=False, cmap="RdBu_r", center=0, ax=ax, vmax=vmax, vmin=-vmax,
                xticklabels=color_names, yticklabels=[f"F{i}" for i in range(B.shape[0])])
    ax.set_title(title)

instruct_results = [r for r in raw_results if r['Model'] == 'Instruct']
fig, axes = plt.subplots(3, 2, figsize=(16, 18))
for i, crit in enumerate(["Social", "Personalized", "Random"]):
    for j, emb in enumerate(["HS", "ST"]):
        m = [r for r in instruct_results if r['Embedding'] == emb and r['Criterion'] == crit]
        if m: plot_b_matrix(m[0]['B'], f"Instruct | {emb} | {crit}", axes[i, j])
        else: axes[i, j].axis('off')
plt.tight_layout()
plt.show()

**Analysis & Conclusions: B Matrix Heatmaps**
Visualizing the heatmaps confirms the numerical similarity metrics. For **HS embeddings**, we see vertical "stripes"—highly correlated weights across all colors for the same feature component. For **ST embeddings**, we see more complex, checkerboard-like patterns, indicating that specific PCA components are being mapped to specific favorite colors. This suggests that the instruction-tuning process might amplify semantic distinctions when using semantic (ST) embeddings but maintains a global persona state when using internal (HS) embeddings.

## 5. Comparative Preference Analysis
Finally, we look at the learned intrinsic preferences ($\delta$) to see how model types and sampling strategies affect results.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
sns.barplot(data=df, x="Color", y="Score (Delta)", hue="Model Type", ax=axes[0], errorbar=None, palette="Set1")
axes[0].set_title("Pretrained vs Instruct")
sns.barplot(data=df, x="Color", y="Score (Delta)", hue="Embedding", ax=axes[1], errorbar=None, palette="Set2")
axes[1].set_title("HS vs ST Embeddings")
sns.barplot(data=df, x="Color", y="Score (Delta)", hue="Criterion", ax=axes[2], errorbar=None, palette="Set3")
axes[2].set_title("Elicitation Criterion Comparison")
for ax in axes: ax.axhline(0, color='black', alpha=0.3)
plt.tight_layout()
plt.show()

**Analysis & Conclusions: Intrinsic Preferences**
The Instruction-tuned model appears to have the same rank-ordering of colors as the Pretrained model, but with **sharper contrasts** (higher absolute delta values). This indicates that the alignment process (RLHF/SFT) tends to polarize preferences. Surprisingly, the choice of elicitation criterion (**Social vs. Personalized**) has a minimal impact on the final mean preferred color, suggesting that the GRUM model is very efficient at identifying the "consensus" preference even when optimizing for individual diversity.

## 6. Global Perspective & Normalization
A comprehensive facet grid showing the final stabilized results.

In [ ]:
g = sns.FacetGrid(df, col="Embedding", row="Model Type", height=4, aspect=1.5, margin_titles=True)
g.map_dataframe(sns.barplot, x="Color", y="Score (Delta)", palette=color_palette, hue="Color", dodge=False, errorbar=None)
g.add_legend(title="Color Item")
plt.subplots_adjust(top=0.9)
g.fig.suptitle("Learned Preferences (Delta) per Model and Embedding")
plt.show()

## 7. Historical Context: The Identifiability Problem
Finally, we reflect on the "unbounded utility drift" observed during early development, which necessitated the sum-to-zero normalization we now use.

In [ ]:
from grums.experiments.utils import load_experiment_results
artifact_path = "/home/lotanamit/grum4llm/artifacts/llm/colors_unbounded"
unbounded_results = {}
for method in ['hs', 'st']:
    path = os.path.join(artifact_path, method)
    if os.path.exists(path):
        unbounded_results[method] = load_experiment_results(path)

if unbounded_results:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    for i, (method, res) in enumerate(unbounded_results.items()):
        social_runs = [k for k in res.keys() if 'social' in k]
        if not social_runs: continue
        sample_key = social_runs[0]
        history = res[sample_key]['history']
        steps = sorted(map(int, history.keys()))
        deltas = np.array([history[str(s)]['grum']['delta'] for s in steps])
        for j in range(deltas.shape[1]):
            color_name = color_names[j]
            axes[i].plot(steps, deltas[:, j], label=color_name, color=color_palette[color_name])
        axes[i].set_title(f'Unbounded Utility Drift ({method.upper()})')
        axes[i].legend(ncol=2)
    plt.show()

**Final Conclusion**
We have demonstrated that GRUMs can be effectively scaled to LLM preference elicitation by utilizing internal and external embedding features. While intrinsic preferences are highly robust, the choice of embedding method creates a fundamental trade-off between capturing "global persona bias" (HS) and "semantic item-driven personalization" (ST). These findings pave the way for more sophisticated elicitation in complex recommendation domains like Laptops and beyond.